The successor is guaranteed to have no more than one child, so we know how to remove it using the two cases for deletion that we have already implemented. Once the successor has been removed, we simply put it in the tree in place of the node to be deleted. The code to handle the third case is shown in Listing 6.14.16.
In Listing 6.14.16 we make use of the helper methods find_successor and splice_out to find and remove the successor. The reason we use splice_out is that it goes directly to the node we want to splice out and makes the right changes. We could call delete recursively, but then we would waste time searching again for the key node.

#62 Python Tutorial for Beginners | Generators
Telusko: https://youtu.be/mziIj4M_uwk?si=VwjUgTW084cSMEcw

In [1]:
class TreeNode:
    def __init__(self, key, value, left=None, right=None, parent=None):
        self.key = key
        self.value = value
        self.left_child = left
        self.right_child = right
        self.parent = parent

    def is_left_child(self):
        #• Checks if this node has a parent (self.parent).
        #• Then checks if that parent's left_child points directly to self 
        #(is self checks for exact object identity in memory).
        #• Returns True if it is a left child, or False / None otherwise.
        return self.parent and self.parent.left_child is self

    def is_right_child(self):
        return self.parent and self.parent.right_child is self

    def is_root(self):
        return not self.parent

    def is_leaf(self):
        return not (self.right_child or self.left_child)

    def has_any_child(self):
        return self.right_child or self.left_child

    def has_children(self):
        return self.right_child and self.left_child

    def replace_value(self, key, value, left, right):
        #Updating the Current Node's Properties
        self.key = key
        self.value = value
        self.left_child = left
        self.right_child = right
        if self.left_child: #if self.left_child:: This checks if a left child actually exists (meaning the left argument passed in wasn't None).
             #If the left child exists, this line tells that child, "Hey, I am your new parent now."   
            self.left_child.parent = self
        if self.right_child:
            self.right_child.parent = self

    def find_successor(self):
        successor = None
        if self.right_child: 
            #Case1: If the node has a right child, then the successor is the smallest key in the right subtree
            successor = self.right_child.find_min()
        else:
            if self.parent: 
                #Case2: If the node has no right child and is the left child of its parent, then the parent is the successor
                if self.is_left_child():# if it is a left child?
                    successor = self.parent
                else: # may be its the right child 
                    #If the node is the right child of its parent,
                    #and itself has no right child, then the successor to this node is the successor of its parent,
                    #excluding this node.
                    self.parent.right_child = None
                    successor = self.parent.find_successor()
                    self.parent.right_child = self
        return successor

    def find_min(self):
        current = self
        while current.left_child:
            current = current.left_child
        return current

    def splice_out(self):# this like delete? used in delete funstion in: class BinarySearchTree 
        if self.is_leaf():
            if self.is_left_child():
                self.parent.left_child = None
            else:
                self.parent.right_child = None
        elif self.has_any_child():
            if self.left_child:
                if self.is_left_child():
                    self.parent.left_child = self.left_child
                else:
                    self.parent.right_child = self.left_child
                self.left_child.parent = self.parent # else its the root
            else:
                if self.is_left_child():
                    self.parent.left_child = self.right_child
                else:
                    self.parent.right_child = self.right_child
                self.right_child.parent = self.parent

    def __iter__(self): #lets Enable Loops
        if self:
            if self.left_child:
                #for elem in self.left_child:: This is a recursive loop.
                #Because self.left_child is also a TreeNode,
                #looping over it automatically triggers its __iter__ method.
                #It dives all the way down to the leftmost leaf node first.
                #yield elem: Forwards every key found in the left subtree up to the main loop.
                #It dives all the way down to the leftmost leaf node first.
                for elem in self.left_child:
                    yield elem
            yield self.key #Yielding the Current Node
            if self.right_child:
                for elem in self.right_child:
                    yield elem


class BinarySearchTree:
    def __init__(self):
        self.root = None
        self.size = 0

    def __len__(self):
        return self.size

    def __iter__(self):
        return self.root.__iter__()

    def put(self, key, value):
        if self.root:
            self._put(key, value, self.root)
        else:
            self.root = TreeNode(key, value)
        self.size = self.size + 1

#Starting at the root of the tree, search the binary tree comparing the new key to the key in the current node.
#If the new key is less than the current node, search the left subtree.
#If the new key is greater than the current node, search the right subtree.
#When there is no left or right child to search, we have found the position in the tree where the new node should be installed.
#To add a node to the tree, create a new TreeNode object and insert the object at the point discovered in the previous step.                    
    def _put(self, key, value, current_node):
        if key < current_node.key:
            if current_node.left_child:
                self._put(key, value, current_node.left_child)
            else:
                current_node.left_child = TreeNode(
                    key, value, parent=current_node
                )
        else:
            if current_node.right_child:
                self._put(key, value, current_node.right_child)
            else:
                current_node.right_child = TreeNode(
                    key, value, parent=current_node
                )


    def __setitem__(self, key, value):
        #This allows us to write Python statements like my_zip_tree['Plymouth'] = 55446, just like a Python dictionary.
        self.put(key, value)

    def get(self, key):
        if self.root:
            result = self._get(key, self.root)
            if result:
                return result.value
        return None

    def _get(self, key, current_node):
        if not current_node:
            return None
        if current_node.key == key:
            return current_node
        elif key < current_node.key:
            return self._get(key, current_node.left_child)
        else:
            return self._get(key, current_node.right_child)

    def __getitem__(self, key):
        return self.get(key)

    def __contains__(self, key):
        #Recall that __contains__ overloads the in operator and allows us to write statements such as
        #"Northfield" in my_zip_tree.
        return bool(self._get(key, self.root))

    def delete(self, key):
        if self.size > 1:
            node_to_remove = self._get(key, self.root)
            if node_to_remove:
                self._delete(node_to_remove)
                self.size = self.size - 1
            else:
                raise KeyError("Error, key not in tree")
        elif self.size == 1 and self.root.key == key:
            self.root = None
            self.size = self.size - 1
        else:
            raise KeyError("Error, key not in tree")

    def _delete(self, current_node):
        if current_node.is_leaf():  # Case1: removing a leaf
            if current_node == current_node.parent.left_child:
                current_node.parent.left_child = None
            else:
                current_node.parent.right_child = None
        elif current_node.has_children():  #Case2: removing a node with two children
            successor = current_node.find_successor()
            successor.splice_out() #is this the actual deletion? is it swapping?
            current_node.key = successor.key
            current_node.value = successor.value
        else:  #Case3: removing a node with one child,we simply promote the child to take the place of its parent.
            if current_node.left_child:
                if current_node.is_left_child():
                    current_node.left_child.parent = current_node.parent
                    current_node.parent.left_child = current_node.left_child
                elif current_node.is_right_child():
                    current_node.left_child.parent = current_node.parent
                    current_node.parent.right_child = current_node.left_child
                else: 
                    #If the current node has no parent, it must be the root.
                    #the function replace_value is described on the: class TreeNode
                    # In this case we will just replace the key, value, left_child, and right_child data
                    #by calling the replace_value method on the root.
                     #It leaves the root node object right where it is in memory,
                    #but overwrites its contents with the contents of its only child.
                    current_node.replace_value(
                        current_node.left_child.key, #get the key of left child
                        current_node.left_child.value, #get value of the right child
                        current_node.left_child.left_child,
                        current_node.left_child.right_child,
                    )
            else:
                if current_node.is_left_child():
                    current_node.right_child.parent = current_node.parent
                    current_node.parent.left_child = current_node.right_child
                elif current_node.is_right_child():
                    current_node.right_child.parent = current_node.parent
                    current_node.parent.right_child = current_node.right_child
                else:
                    current_node.replace_value(
                        current_node.right_child.key,
                        current_node.right_child.value,
                        current_node.right_child.left_child,
                        current_node.right_child.right_child,
                    )

    def __delitem__(self, key):
        #It is a wrapper method that allows us to remove a key from the map by writing del my_zip_tree["NYC"].
        self.delete(key)


my_tree = BinarySearchTree()
my_tree["a"] = "a"
my_tree["q"] = "quick"
my_tree["b"] = "brown"
my_tree["f"] = "fox"
my_tree["j"] = "jumps"
my_tree["o"] = "over"
my_tree["t"] = "the"
my_tree["l"] = "lazy"
my_tree["d"] = "dog"

print(my_tree["q"])
print(my_tree["l"])
print("There are {} items in this tree".format(len(my_tree)))
my_tree.delete("a")
print("There are {} items in this tree".format(len(my_tree)))

for node in my_tree:
    print(my_tree[node], end=" ")
print()


quick
lazy
There are 9 items in this tree
There are 8 items in this tree
brown dog fox jumps lazy over quick the 


#### Playground

In [11]:
not None, 3 is 6/2 ,3.0 is 6/2, 3 == 6/2, False == None

(True, False, True, True, False)

In [12]:
3 == 6/2

True

In [4]:
8-3


5

In [5]:
class TreeNode:
    def __init__(self, key, value, left=None, right=None, parent=None):
        self.key = key
        self.value = value
        self.left_child = left
        self.right_child = right
        self.parent = parent

    def is_left_child(self):
        """Checks if this node is the left child of its parent."""
        return self.parent and self.parent.left_child is self

    def is_right_child(self):
        """Checks if this node is the right child of its parent."""
        return self.parent and self.parent.right_child is self

    def is_root(self):
        """Checks if this node is the absolute root of the tree."""
        return not self.parent

    def is_leaf(self):
        """Checks if this node has no children at all."""
        return not (self.right_child or self.left_child)

    def has_any_child(self):
        """Checks if this node has at least one child."""
        return self.right_child or self.left_child

    def has_children(self):
        """Checks if this node has both a left and a right child."""
        return self.right_child and self.left_child

    def replace_value(self, key, value, left, right):
        """Updates the internal components of a node and resets child parent flags."""
        self.key = key
        self.value = value
        self.left_child = left
        self.right_child = right
        if self.left_child:
            self.left_child.parent = self
        if self.right_child:
            self.right_child.parent = self

    def find_successor(self):
        """Locates the next largest node in order (used during deletion)."""
        successor = None
        if self.right_child:
            successor = self.right_child.find_min()
        else:
            if self.parent:
                if self.is_left_child():
                    successor = self.parent
                else:
                    self.parent.right_child = None
                    successor = self.parent.find_successor()
                    self.parent.right_child = self
        return successor

    def find_min(self):
        """Helper that travels all the way left to find the minimum value."""
        current = self
        while current.left_child:
            current = current.left_child
        return current

    def splice_out(self):
        """Bypasses a node by linking its parent directly to its child."""
        if self.is_leaf():
            if self.is_left_child():
                self.parent.left_child = None
            else:
                self.parent.right_child = None
        elif self.has_any_child():
            if self.left_child:
                if self.is_left_child():
                    self.parent.left_child = self.left_child
                else:
                    self.parent.right_child = self.left_child
                self.left_child.parent = self.parent
            else:
                if self.is_left_child():
                    self.parent.left_child = self.right_child
                else:
                    self.parent.right_child = self.right_child
                self.right_child.parent = self.parent

    def __iter__(self):
        """Yields keys in ascending order using Inorder Traversal loops."""
        if self:
            if self.left_child:
                for elem in self.left_child:
                    yield elem
            yield self.key
            if self.right_child:
                for elem in self.right_child:
                    yield elem


class BinarySearchTree:
    def __init__(self):
        self.root = None
        self.size = 0

    def __len__(self):
        return self.size

    def __iter__(self):
        return self.root.__iter__()

    def put(self, key, value):
        """Inserts a key-value pair. Installs as root if tree is empty."""
        if self.root:
            # Captures whether a new node was created or an old one was updated
            is_new_node = self._put(key, value, self.root)
            if is_new_node:
                self.size = self.size + 1
        else:
            self.root = TreeNode(key, value)
            self.size = self.size + 1

    def _put(self, key, value, current_node):
        """Recursive helper that positions keys or updates existing values."""
        # FIX: Handle Duplicate Keys Inline
        if key == current_node.key:
            current_node.value = value
            return False  # Flags that NO new node was added to the tree structure
            
        elif key < current_node.key:
            if current_node.left_child:
                return self._put(key, value, current_node.left_child)
            else:
                current_node.left_child = TreeNode(key, value, parent=current_node)
                return True  # New node created
        else:
            if current_node.right_child:
                return self._put(key, value, current_node.right_child)
            else:
                current_node.right_child = TreeNode(key, value, parent=current_node)
                return True  # New node created

    def __setitem__(self, key, value):
        """Enables standard dictionary style syntax: tree[key] = value"""
        self.put(key, value)

    def get(self, key):
        """Retrieves values by key. Returns None if key is absent."""
        if self.root:
            result = self._get(key, self.root)
            if result:
                return result.value
        return None

    def _get(self, key, current_node):
        """Recursive node retriever helper."""
        if not current_node:
            return None
        if current_node.key == key:
            return current_node
        elif key < current_node.key:
            return self._get(key, current_node.left_child)
        else:
            return self._get(key, current_node.right_child)

    def __getitem__(self, key):
        """Enables item lookup syntax: value = tree[key]"""
        return self.get(key)

    def __contains__(self, key):
        """Enables evaluation checks: 'key in tree'"""
        return bool(self._get(key, self.root))

    def delete(self, key):
        """Main deletion manager routing framework size updates."""
        if self.size > 1:
            node_to_remove = self._get(key, self.root)
            if node_to_remove:
                self._delete(node_to_remove)
                self.size = self.size - 1
            else:
                raise KeyError("Error, key not in tree")
        elif self.size == 1 and self.root.key == key:
            self.root = None
            self.size = self.size - 1
        else:
            raise KeyError("Error, key not in tree")

    def _delete(self, current_node):
        """Handles structural node extractions based on child layout."""
        if current_node.is_leaf():  # Case 1: Removing a leaf node
            if current_node == current_node.parent.left_child:
                current_node.parent.left_child = None
            else:
                current_node.parent.right_child = None
                
        elif current_node.has_children():  # Case 2: Node has two children
            successor = current_node.find_successor()
            successor.splice_out()
            current_node.key = successor.key
            current_node.value = successor.value
            
        else:  # Case 3: Node has exactly one child (Restored Cutoff Segment)
            if current_node.left_child:
                if current_node.is_left_child():
                    current_node.left_child.parent = current_node.parent
                    current_node.parent.left_child = current_node.left_child
                elif current_node.is_right_child():
                    current_node.left_child.parent = current_node.parent
                    current_node.parent.right_child = current_node.left_child
                else: # Current node is the root with a left child
                    current_node.replace_value(
                        current_node.left_child.key,
                        current_node.left_child.value,
                        current_node.left_child.left_child,
                        current_node.left_child.right_child,
                    )
            else: # Current node has a right child
                if current_node.is_left_child():
                    current_node.right_child.parent = current_node.parent
                    current_node.parent.left_child = current_node.right_child
                elif current_node.is_right_child():
                    current_node.right_child.parent = current_node.parent
                    current_node.parent.right_child = current_node.right_child
                else: # Current node is the root with a right child
                    current_node.replace_value(
                        current_node.right_child.key,
                        current_node.right_child.value,
                        current_node.right_child.left_child,
                        current_node.right_child.right_child,
                    )


# ==========================================
# Test Verification Loop
# ==========================================
bst = BinarySearchTree()

# Insert unique key-value items
bst[70] = "Apple"
bst[31] = "Banana"
bst[93] = "Cherry"

print("--- Duplicate Key Handling Test ---")
print(f"Initial 31 Value: '{bst[31]}'")
print(f"Initial Tree Size: {len(bst)}")

# Trigger duplicate insertion update
bst[31] = "Blueberry"

print(f"Updated 31 Value: '{bst[31]}' (Should be Blueberry)")
print(f"Updated Tree Size: {len(bst)} (Should still be 3)")


--- Duplicate Key Handling Test ---
Initial 31 Value: 'Banana'
Initial Tree Size: 3
Updated 31 Value: 'Blueberry' (Should be Blueberry)
Updated Tree Size: 3 (Should still be 3)


In [6]:
class TreeNode:
    def __init__(self, key, value, left=None, right=None, parent=None):
        self.key = key
        self.value = value
        self.left_child = left
        self.right_child = right
        self.parent = parent

    def is_left_child(self):
        """Checks if this node is the left child of its parent."""
        return self.parent and self.parent.left_child is self

    def is_right_child(self):
        """Checks if this node is the right child of its parent."""
        return self.parent and self.parent.right_child is self

    def is_root(self):
        """Returns True if the node has no parent (it is the root)."""
        return not self.parent

    def is_leaf(self):
        """Returns True if the node has zero children."""
        return not (self.right_child or self.left_child)

    def has_any_child(self):
        """Returns True if the node has at least one child."""
        return self.right_child or self.left_child

    def has_children(self):
        """Returns True if the node has both left and right children."""
        return self.right_child and self.left_child

    def replace_value(self, key, value, left, right):
        """Updates internal node values and resets parent properties for active children."""
        self.key = key
        self.value = value
        self.left_child = left
        self.right_child = right
        if self.left_child:
            self.left_child.parent = self
        if self.right_child:
            self.right_child.parent = self

    def find_successor(self):
        """Locates the next largest node in order (used during two-child deletion)."""
        successor = None
        if self.right_child:
            successor = self.right_child.find_min()
        else:
            if self.parent:
                if self.is_left_child():
                    successor = self.parent
                else:
                    self.parent.right_child = None
                    successor = self.parent.find_successor()
                    self.parent.right_child = self
        return successor

    def find_min(self):
        """Finds the minimum key in the subtree by going all the way left."""
        current = self
        while current.left_child:
            current = current.left_child
        return current

    def splice_out(self):
        """Bypasses a node by linking its parent directly to its single child."""
        if self.is_leaf():
            if self.is_left_child():
                self.parent.left_child = None
            else:
                self.parent.right_child = None
        elif self.has_any_child():
            if self.left_child:
                if self.is_left_child():
                    self.parent.left_child = self.left_child
                else:
                    self.parent.right_child = self.left_child
                self.left_child.parent = self.parent
            else:
                if self.is_left_child():
                    self.parent.left_child = self.right_child
                else:
                    self.parent.right_child = self.right_child
                self.right_child.parent = self.parent

    def __iter__(self):
        """Yields keys in ascending sorted order using an Inorder Traversal loop."""
        if self:
            if self.left_child:
                for elem in self.left_child:
                    yield elem
            yield self.key
            if self.right_child:
                for elem in self.right_child:
                    yield elem


class BinarySearchTree:
    def __init__(self):
        self.root = None
        self.size = 0

    def __len__(self):
        return self.size

    def __iter__(self):
        return self.root.__iter__()

    def put(self, key, value):
        """Inserts a key-value pair into the BST."""
        if self.root:
            is_new_node = self._put(key, value, self.root)
            if is_new_node:
                self.size = self.size + 1
        else:
            self.root = TreeNode(key, value)
            self.size = self.size + 1

    def _put(self, key, value, current_node):
        """Recursive helper that fixes duplicate keys by replacing values inline."""
        if key == current_node.key:
            current_node.value = value  # Overwrite old value
            return False  # No new node structure added
            
        elif key < current_node.key:
            if current_node.left_child:
                return self._put(key, value, current_node.left_child)
            else:
                current_node.left_child = TreeNode(key, value, parent=current_node)
                return True
        else:
            if current_node.right_child:
                return self._put(key, value, current_node.right_child)
            else:
                current_node.right_child = TreeNode(key, value, parent=current_node)
                return True

    def __setitem__(self, key, value):
        """Enables tree[key] = value syntax."""
        self.put(key, value)

    def get(self, key):
        """Fetches the value of a key. Returns None if key is absent."""
        if self.root:
            result = self._get(key, self.root)
            if result:
                return result.value
        return None

    def _get(self, key, current_node):
        """Recursive search helper to retrieve nodes."""
        if not current_node:
            return None
        if current_node.key == key:
            return current_node
        elif key < current_node.key:
            return self._get(key, current_node.left_child)
        else:
            return self._get(key, current_node.right_child)

    def __getitem__(self, key):
        """Enables value = tree[key] syntax."""
        return self.get(key)

    def __contains__(self, key):
        """Enables 'key in tree' checks."""
        return bool(self._get(key, self.root))

    def delete(self, key):
        """Deletes a node matching the target key."""
        if self.size > 1:
            node_to_remove = self._get(key, self.root)
            if node_to_remove:
                self._delete(node_to_remove)
                self.size = self.size - 1
            else:
                raise KeyError("Error, key not in tree")
        elif self.size == 1 and self.root.key == key:
            self.root = None
            self.size = self.size - 1
        else:
            raise KeyError("Error, key not in tree")

    def _delete(self, current_node):
        """Structural node modifier framework for deletions."""
        if current_node.is_leaf():  # Case 1: Removing a leaf node
            if current_node == current_node.parent.left_child:
                current_node.parent.left_child = None
            else:
                current_node.parent.right_child = None
                
        elif current_node.has_children():  # Case 3: Node has two children
            successor = current_node.find_successor()
            successor.splice_out()
            current_node.key = successor.key
            current_node.value = successor.value
            
        else:  # Case 2: Node has exactly one child
            if current_node.left_child:
                if current_node.is_left_child():
                    current_node.left_child.parent = current_node.parent
                    current_node.parent.left_child = current_node.left_child
                elif current_node.is_right_child():
                    current_node.left_child.parent = current_node.parent
                    current_node.parent.right_child = current_node.left_child
                else:
                    current_node.replace_value(
                        current_node.left_child.key,
                        current_node.left_child.value,
                        current_node.left_child.left_child,
                        current_node.left_child.right_child,
                    )
            else:
                if current_node.is_left_child():
                    current_node.right_child.parent = current_node.parent
                    current_node.parent.left_child = current_node.right_child
                elif current_node.is_right_child():
                    current_node.right_child.parent = current_node.parent
                    current_node.parent.right_child = current_node.right_child
                else:
                    current_node.replace_value(
                        current_node.right_child.key,
                        current_node.right_child.value,
                        current_node.right_child.left_child,
                        current_node.right_child.right_child,
                    )

    def display(self):
        """Public method to start printing the tree visual layout."""
        if self.root is None:
            print("🌲 The tree is completely empty.")
        else:
            print("\n--- Visual BST Layout (Read Sideways: Left is Left Child, Right is Right Child) ---")
            self._display(self.root, prefix="", is_left=None)
            print("-" * 75 + "\n")

    def _display(self, current_node, prefix="", is_left=None):
        """Recursive helper that renders the tree map using branch strings."""
        if current_node is not None:
            # 1. Process the right subtree first (prints at the top of the terminal screen)
            if current_node.right_child:
                next_prefix = prefix + ("│   " if is_left == True else "    ")
                self._display(current_node.right_child, next_prefix, is_left=False)

            # 2. Print the current node with its appropriate connector symbol
            if is_left is None:
                # Root node symbol
                branch_symbol = "── "
            elif is_left:
                branch_symbol = "└── (L) "
            else:
                branch_symbol = "┌── (R) "

            print(f"{prefix}{branch_symbol}[{current_node.key}: {current_node.value}]")

            # 3. Process the left subtree last (prints at the bottom of the screen)
            if current_node.left_child:
                next_prefix = prefix + ("    " if is_left == False else "│   ")
                self._display(current_node.left_child, next_prefix, is_left=True)


    def __delitem__(self, key):
        """Enables del tree[key] syntax."""
        self.delete(key)


# =====================================================================
# 3. VERIFICATION EXECUTION TRACE
# =====================================================================

my_tree = BinarySearchTree()
my_tree["a"] = "a"
my_tree["q"] = "quick"
my_tree["b"] = "brown"
my_tree["f"] = "fox"
my_tree["j"] = "jumps"
my_tree["o"] = "over"
my_tree["t"] = "the"
my_tree["l"] = "lazy"
my_tree["d"] = "dog"

print("--- 🔍 Key Item Retrievals ---")
print(f"Value at ['q']: {my_tree['q']}")
print(f"Value at ['l']: {my_tree['l']}")

print("\n--- 📊 Size Verification Tracking ---")
print("There are {} items in this tree".format(len(my_tree)))
my_tree.delete("a")
print("There are {} items in this tree (After deleting 'a')".format(len(my_tree)))

print("\n--- 🔤 Sorted Inorder Content Loop ---")
for node in my_tree:
    print(my_tree[node], end=" ")
print()



# Create a fresh tree instance
tree = BinarySearchTree()

# Insert raw keys to build a multi-level layout
tree["m"] = "Medium"
tree["f"] = "Fox"
tree["t"] = "The"
tree["b"] = "Brown"
tree["j"] = "Jumps"

# Display the layout
tree.display()


--- 🔍 Key Item Retrievals ---
Value at ['q']: quick
Value at ['l']: lazy

--- 📊 Size Verification Tracking ---
There are 9 items in this tree
There are 8 items in this tree (After deleting 'a')

--- 🔤 Sorted Inorder Content Loop ---
brown dog fox jumps lazy over quick the 

--- Visual BST Layout (Read Sideways: Left is Left Child, Right is Right Child) ---
    ┌── (R) [t: The]
── [m: Medium]
│   │   ┌── (R) [j: Jumps]
│   └── (L) [f: Fox]
│   │   └── (L) [b: Brown]
---------------------------------------------------------------------------



In [8]:
bool(7)

True

In [13]:
import time

class TraceTreeNode:
    def __init__(self, key, left=None, right=None):
        self.key = key
        self.left_child = left
        self.right_child = right

    def __iter__(self):
        """
        Traces the execution step-by-step to show how 
        generators pause and wake up recursively.
        """
        print(f"▶️  [Node {self.key}] Entered __iter__ method.")
        time.sleep(0.3)
        
        # 1. Traverse Left Subtree
        if self.left_child:
            print(f"⏬ [Node {self.key}] Pausing to enter recursive loop for Left Child: Node {self.left_child.key}")
            time.sleep(0.2)
            
            for elem in self.left_child:
                print(f"🔄 [Node {self.key}] Intercepted value '{elem}' from Left child loop. Forwarding it up...")
                time.sleep(0.2)
                yield elem
                print(f"🌤️  [Node {self.key}] Woke up inside Left child loop after forwarding '{elem}'. Checking for more left elements...")
                time.sleep(0.2)
                
            print(f"✅ [Node {self.key}] Left child loop is completely exhausted.")
        else:
            print(f"ℹ️  [Node {self.key}] No left child discovered.")

        # 2. Visit Current Root
        print(f"🎯 [Node {self.key}] Yielding its own key: {self.key}")
        time.sleep(0.2)
        yield self.key
        
        print(f"🌤️  [Node {self.key}] Woke up right after yielding its own key ({self.key}). Moving to check the Right branch...")
        time.sleep(0.2)

        # 3. Traverse Right Subtree
        if self.right_child:
            print(f"⏬ [Node {self.key}] Pausing to enter recursive loop for Right Child: Node {self.right_child.key}")
            time.sleep(0.2)
            
            for elem in self.right_child:
                print(f"🔄 [Node {self.key}] Intercepted value '{elem}' from Right child loop. Forwarding it up...")
                time.sleep(0.2)
                yield elem
                print(f"🌤️  [Node {self.key}] Woke up inside Right child loop after forwarding '{elem}'. Checking for more right elements...")
                time.sleep(0.2)
                
            print(f"✅ [Node {self.key}] Right child loop is completely exhausted.")
        else:
            print(f"ℹ️  [Node {self.key}] No right child discovered.")

        print(f"🛑 [Node {self.key}] __iter__ method has finished execution. Exiting frame.")


# =====================================================================
# Execution Setup
# =====================================================================

# Build a small balanced tree:
#         50
#       /    \
#      30    70

node_30 = TraceTreeNode(30)
node_70 = TraceTreeNode(70)
root_node = TraceTreeNode(50, left=node_30, right=node_70)

print("=====================================================================")
print("🚀 STARTING ITERATION PROCESS ON ROOT (NODE 50)")
print("=====================================================================\n")

# Triggering the master loop
for item in root_node:
    print(f"\n🎉 [MAIN LOOP] Successfully received final yielded output value: >>> {item} <<<\n")
    print("-" * 69)

print("\n=====================================================================")
print("🎉 ITERATION COMPLETELY FINISHED!")
print("=====================================================================")


🚀 STARTING ITERATION PROCESS ON ROOT (NODE 50)

▶️  [Node 50] Entered __iter__ method.
⏬ [Node 50] Pausing to enter recursive loop for Left Child: Node 30
▶️  [Node 30] Entered __iter__ method.
ℹ️  [Node 30] No left child discovered.
🎯 [Node 30] Yielding its own key: 30
🔄 [Node 50] Intercepted value '30' from Left child loop. Forwarding it up...

🎉 [MAIN LOOP] Successfully received final yielded output value: >>> 30 <<<

---------------------------------------------------------------------
🌤️  [Node 50] Woke up inside Left child loop after forwarding '30'. Checking for more left elements...
🌤️  [Node 30] Woke up right after yielding its own key (30). Moving to check the Right branch...
ℹ️  [Node 30] No right child discovered.
🛑 [Node 30] __iter__ method has finished execution. Exiting frame.
✅ [Node 50] Left child loop is completely exhausted.
🎯 [Node 50] Yielding its own key: 50

🎉 [MAIN LOOP] Successfully received final yielded output value: >>> 50 <<<

------------------------------

At first glance, this code looks like a standard for loop rather than a recursive function because you dont explicitly see a line that reads self.__iter__().
However, the recursion is hidden inside the for ... in loop syntax itself. In Python, whenever you write for elem in X:, Python automatically goes behind the scenes, finds the __iter__ method of object X, and executes it.
Because self.left_child and self.right_child are also instances of the TreeNode class, looping over them triggers their own __iter__ method!


 **🔍 Breaking Down the Hidden Recursion** 
 
Let's look at exactly how Python interprets this code mechanically:

def __iter__(self):
    if self:
        if self.left_child:
            # 💥 THIS LINE IS THE RECURSION!
            # Because self.left_child is a TreeNode, this line tells Python:
            # "Go run the __iter__ method on my left child node right now."
            for elem in self.left_child:  
                yield elem  # Forwards the values up the call stack
                
        yield self.key  # Visits the current parent node (Root)
        
        if self.right_child:
            # 💥 THIS LINE IS ALSO RECURSION!
            # Triggers the __iter__ method on the right child node.
            for elem in self.right_child:
                yield elem

------------------------------
**🗺️ The Step-by-Step Flow of Generators (yield)**
Unlike a normal recursive function that computes a final value and returns it all at once, this method uses yield, making it a Generator.
When a function uses yield, it doesn't stop executing completely when it returns a value; instead, it pauses its state and hands a single item up to the caller. When the caller asks for the next item, the function wakes up right where it left off.
Let's trace how a small tree handles this execution stack:

       [50]
      /    \
   [30]    [70]


   1. You start a loop on the root node 50 via for item in tree:.
   2. Node 50 enters its __iter__. It sees a left child, so it executes for elem in self.left_child:.
   3. The Recursive Jump: Python pauses Node 50 and jumps down to execute __iter__ on Node 30.
   4. Node 30 checks for a left child. It has none (None).
   5. Node 30 moves to its next line: yield self.key. It hands the value 30 up to the loop on Node 50.
   6. Node 50 receives 30 inside its internal for elem loop, and immediately executes yield elem, passing 30 all the way up to your main outer print loop.
   7. Node 30 checks for a right child. It has none, so its function finishes.
   8. Control returns back to Node 50. Having finished its left-child loop, Node 50 moves to its next line: yield self.key. It yields 50 up to your main loop.
   9. Node 50 then hits its right-child section and executes for elem in self.right_child:.
   10. The Next Recursive Jump: Python pauses Node 50 and jumps down to execute __iter__ on Node 70, which yields 70 cleanly.

Final output order: 30 -> 50 -> 70 (Perfect Inorder Traversal!)

**💡 Modern Python Shortcut**
In modern Python (Python 3.3+), this exact loop can be written even cleaner using the yield from statement, which removes the need for the internal for loop entirely:

def __iter__(self):
    if self:
        if self.left_child:
            yield from self.left_child
        yield self.key
        if self.right_child:
            yield from self.right_child

Both syntaxes perform the exact same hidden recursive operation!
If you want to see how this fits into your main runtime loop, let me know:

* Would you like to run a complete trace script that prints messages when it pauses and wakes up to see the generator timeline?
* Do you want to try converting this generator structure into a Preorder traversal (__iter__) instead?


